# Sentyment newsów NVDA 2017–2026: GDELT → finbert-tone → cechy ML

**Źródło danych:** GDELT GKG — darmowe archiwum newsów od 2013  
**Model:** `yiyanghkust/finbert-tone`  
**Wyjście:** `nvda_sentiment_features.csv` — 5 cech dziennych do XGBoost/RF

### Strategia źródeł zależna od okresu
- **2017–2021:** szerokie pokrycie (17 źródeł) + Wayback Machine dla martwych linków  
- **2022–2026:** wąskie pokrycie (6 kluczowych źródeł), linki żyją bezpośrednio

### Szacowany czas działania
~4–6h dla pełnego zakresu 2017–2026 (checkpoint co miesiąc — można przerywać)


## 1. Importy i konfiguracja

In [2]:
import requests
import zipfile
import io
import re
import os
import time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from bs4 import BeautifulSoup

# 2017–2021: szerokie — dużo martwych linków, potrzebujemy Wayback Machine
SOURCES_HISTORICAL = [
    "reuters", "bloomberg", "cnbc", "wsj", "marketwatch",
    "seekingalpha", "forbes", "benzinga", "barrons",
    "motleyfool", "zacks", "techcrunch", "theverge",
    "wired", "tomshardware", "nvidia", "ft.com"
]

# 2022–2026: wąskie — tylko analityczne, linki żyją bezpośrednio
# Mniej śmieci, wyższy SNR (signal-to-noise ratio)

# ale mało artykółów, więc usuwam to
SOURCES_RECENT = [
    "reuters", "bloomberg", "cnbc", "seekingalpha", "wsj", "barrons"
]

def get_sources(date: datetime) -> list:
    return SOURCES_HISTORICAL

OUTPUT_RAW  = "nvda_news_raw.csv"          # tytuły ze slugów URL
OUTPUT_SENT = "nvda_sentiment_features.csv" # finalne cechy
CHECKPOINT  = "nvda_checkpoint.txt"         # pobrane miesiące

print("Konfiguracja OK")
print(f"Źródła historyczne (2017-2021): {len(SOURCES_HISTORICAL)}")
print(f"Źródła najnowsze  (2022-2026): {len(SOURCES_RECENT)}")


Konfiguracja OK
Źródła historyczne (2017-2021): 17
Źródła najnowsze  (2022-2026): 6


## 2. Pobieranie danych GDELT GKG (2017–2026)

Pliki GKG dostępne dzień po dniu od 2015. Każdy ZIP to ~5–50MB.  
Filtrujemy po źródle i URL zawierającym "nvidia/nvda".  
Checkpoint zapisuje postęp — można przerywać i wznawiać.


In [7]:
with open("nvda_checkpoint.txt") as f:
    print(f.read())

df = pd.read_csv("nvda_news_raw.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Łącznie artykułów: {len(df)}")
print(f"Zakres: {df['date'].min().date()} → {df['date'].max().date()}")

print("\nTop źródła ogółem:")
print(df["source"].value_counts().head(15).to_string())

print("\nCzy jest Reuters?")
reuters = df[df["source"].str.contains("reuters", case=False, na=False)]
print(f"Artykułów z Reuters: {len(reuters)}")
print(reuters[["date", "title_slug", "source"]].head(5).to_string())

print("\nPer rok:")
print(df.groupby(["year", "source"])["url"].count()
      .reset_index()
      .sort_values(["year", "url"], ascending=[True, False])
      .groupby("year").head(3)
      .to_string())

2017-01
2017-02
2017-03
2017-04
2017-05
2017-06
2017-07
2017-08
2017-09
2017-10
2017-11
2017-12
2018-01
2018-02
2018-03
2018-04
2018-05
2018-06
2018-07
2018-08
2018-09
2018-10
2018-11
2018-12
2019-01
2019-02
2019-03
2019-04
2019-05
2019-06
2019-07
2019-08
2019-09
2019-10
2019-11
2019-12
2020-01
2020-02
2020-03
2020-04
2020-05
2020-06
2020-07
2020-08
2020-09
2020-10
2020-11
2020-12
2021-01
2021-02
2021-03
2021-04
2021-05
2021-06
2021-07
2021-08
2021-09
2021-10
2021-11

Łącznie artykułów: 2131
Zakres: 2017-01-02 → 2021-11-25

Top źródła ogółem:
source
marketwatch.com                        380
forbes.com                             272
reuters.com                            246
benzinga.com                           233
seekingalpha.com                       223
cnbc.com                               117
nvidia.com                             101
barrons.com                             98
reuters.com;reuters.com                 93
theverge.com                            69
tomshardware.c

In [10]:
# Sprawdź od kiedy masz dane
df = pd.read_csv("nvda_news_raw.csv")
df["date"] = pd.to_datetime(df["date"])
print(f"Masz dane od: {df['date'].min().date()} do: {df['date'].max().date()}")

# Usuń z checkpointu wszystkie miesiące >= 2022 żeby pobrać je ponownie
with open("nvda_checkpoint.txt") as f:
    lines = [l.strip() for l in f if l.strip()]

to_keep   = [l for l in lines if int(l[:4]) <= 2021]
to_rerun  = [l for l in lines if int(l[:4]) >= 2022]

with open("nvda_checkpoint.txt", "w") as f:
    f.write("\n".join(to_keep) + "\n" if to_keep else "")

print(f"\nZachowane w checkpoincie ({len(to_keep)} miesięcy): {to_keep[:3]}...")
print(f"Do ponownego pobrania ({len(to_rerun)} miesięcy): {to_rerun[:3]}...")

Masz dane od: 2017-01-02 do: 2023-01-27

Zachowane w checkpoincie (60 miesięcy): ['2017-01', '2017-02', '2017-03']...
Do ponownego pobrania (13 miesięcy): ['2022-01', '2022-02', '2022-03']...


In [11]:
def fetch_one_day(date: datetime) -> pd.DataFrame:
    date_str = date.strftime("%Y%m%d")
    url = f"http://data.gdeltproject.org/gkg/{date_str}.gkg.csv.zip"
    sources = get_sources(date)

    try:
        r = requests.get(url, timeout=60)
        if r.status_code != 200:
            return pd.DataFrame()

        z = zipfile.ZipFile(io.BytesIO(r.content))
        df = pd.read_csv(
            z.open(z.namelist()[0]),
            sep="\t", header=0,
            usecols=["DATE", "SOURCES", "SOURCEURLS", "THEMES", "ORGANIZATIONS"],
            on_bad_lines="skip",
            encoding_errors="replace"
        )

        mask_nvidia = (
            df["ORGANIZATIONS"].str.contains("nvidia", case=False, na=False) |
            df["SOURCEURLS"].str.contains("nvidia|nvda", case=False, na=False) |
            df["THEMES"].str.contains("nvidia", case=False, na=False)
        )
        mask_source = df["SOURCES"].str.contains(
            "|".join(sources), case=False, na=False
        )
        df = df[mask_nvidia & mask_source].copy()
        if df.empty:
            return pd.DataFrame()

        df["url_list"] = df["SOURCEURLS"].str.split(",")
        df = df.explode("url_list")
        df = df[df["url_list"].str.contains("nvidia|nvda", case=False, na=False)]

        df["title_slug"] = (
            df["url_list"]
            .str.extract(r"/([^/]+?)(?:\.html?|\.asp\w*|/$|$)")[0]
            .str.replace(r"[-_]", " ", regex=True)
            .str.replace(r"\?.*", "", regex=True)
            .str.strip()
        )
        df["date"]   = pd.to_datetime(date_str, format="%Y%m%d")
        df["source"] = df["SOURCES"]
        df["year"]   = date.year

        return (df[["date", "year", "title_slug", "source", "url_list"]]
                .rename(columns={"url_list": "url"})
                .dropna(subset=["date", "title_slug"])
                .query("title_slug.str.len() > 10"))

    except Exception as e:
        return pd.DataFrame()


def load_checkpoint() -> set:
    if not os.path.exists(CHECKPOINT):
        return set()
    with open(CHECKPOINT) as f:
        return set(line.strip() for line in f if line.strip())

def save_checkpoint(month_key: str):
    with open(CHECKPOINT, "a") as f:
        f.write(month_key + "\n")

def append_raw(records: list):
    if not records:
        return
    df_new = pd.DataFrame(records)
    df_new["date"] = pd.to_datetime(df_new["date"])
    if os.path.exists(OUTPUT_RAW):
        df_old = pd.read_csv(OUTPUT_RAW)
        df_old["date"] = pd.to_datetime(df_old["date"])  # string → Timestamp
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new
    df_all = df_all.drop_duplicates(subset=["url"]).sort_values("date")
    df_all.to_csv(OUTPUT_RAW, index=False)


# ── Główna pętla pobierania ───────────────────────────────────────────────────
def run_full_download(start="2017-01-01", end="2026-01-01"):
    done    = load_checkpoint()
    current = datetime.strptime(start, "%Y-%m-%d")
    end_dt  = datetime.strptime(end,   "%Y-%m-%d")
    batch   = []

    print(f"Start pobierania: {start} → {end}")
    print(f"Już pobrano: {len(done)} miesięcy")
    print("-" * 50)

    while current < end_dt:
        # Wyznacz koniec miesiąca
        if current.month == 12:
            month_end = datetime(current.year + 1, 1, 1)
        else:
            month_end = datetime(current.year, current.month + 1, 1)
        month_end = min(month_end, end_dt)
        month_key = current.strftime("%Y-%m")

        if month_key in done:
            current = month_end
            continue

        print(f"\n[{month_key}] ", end="", flush=True)
        month_records = []

        day = current
        while day < month_end:
            df_day = fetch_one_day(day)
            if not df_day.empty:
                month_records.extend(df_day.to_dict("records"))
                print(".", end="", flush=True)
            else:
                print("_", end="", flush=True)
            day += timedelta(days=1)
            time.sleep(0.3)

        print(f" → {len(month_records)} artykułów")
        batch.extend(month_records)

        # Zapisuj co miesiąc
        append_raw(batch)
        batch = []
        save_checkpoint(month_key)
        current = month_end

    print("\n" + "="*50)
    print("Pobieranie zakończone!")
    if os.path.exists(OUTPUT_RAW):
        df = pd.read_csv(OUTPUT_RAW)
        print(f"Łącznie artykułów: {len(df)}")
        print(f"Zakres: {df['date'].min()} → {df['date'].max()}")
        print("\nArtykuły per rok:")
        print(df.groupby("year")["url"].count().to_string())


# Pobieranie 
run_full_download()


Start pobierania: 2017-01-01 → 2026-01-01
Już pobrano: 60 miesięcy
--------------------------------------------------

[2022-01] ___._.___.._...________..___.__ → 19 artykułów

[2022-02] _____._....___.._____._..... → 27 artykułów

[2022-03] .__..._..___._._.._._.._._..___ → 26 artykułów

[2022-04] _..____.__.._.___....__.____._ → 20 artykułów

[2022-05] ___...___....______.__...._.... → 36 artykułów

[2022-06] .___________..__.___....__.._. → 14 artykułów

[2022-07] ..____.____._._...._____._..___ → 20 artykułów

[2022-08] .___.....__.___.._.._......__.. → 37 artykułów

[2022-09] .._._._..__..__.._.._..___..._ → 58 artykułów

[2022-10] __..._._...__.___...._._...._._ → 34 artykułów

[2022-11] _._.__..._.._.....__....____.. → 40 artykułów

[2022-12] ____.._____.______.__..__.._.._ → 15 artykułów

[2023-01] _..__.____.._.___...__..._.__._ → 27 artykułów

[2023-02] .__.____..__._______..._.__. → 17 artykułów

[2023-03] _..__..._.___.____...._.._...._ → 34 artykułów

[2023-04] _.._.__.___

## 3. Pobieranie prawdziwych tytułów

**Strategia:**
- Zawsze próbuje bezpośredni URL (og:title)
- Dla lat 2017–2021: fallback na Wayback Machine dla martwych linków
- Wynik zapisywany do **`nvda_news_enriched.csv`** — surowy plik `nvda_news_raw.csv` pozostaje nienaruszony jako backup

Checkpoint tytułów: pomija artykuły które już są w `nvda_news_enriched.csv` — można przerywać i wznawiać.


In [ ]:
OUTPUT_ENRICHED = "nvda_news_enriched.csv"  # backup raw, enriched osobno

def get_title_direct(url: str) -> str:
    """Pobiera tytuł ze strony (og:title lub <title>)."""
    try:
        r = requests.get(url, timeout=8,
                         headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(r.text, "html.parser")
        og = soup.find("meta", property="og:title")
        if og and og.get("content") and len(og["content"].strip()) > 15:
            return og["content"].strip()
        if soup.title and soup.title.string and len(soup.title.string.strip()) > 15:
            return soup.title.string.strip()
    except:
        pass
    return ""

def get_title_wayback(url: str, date_str: str) -> str:
    """Pobiera tytuł z archiwum Wayback Machine (fallback dla martwych linków)."""
    try:
        api = f"http://archive.org/wayback/available?url={url}&timestamp={date_str}"
        r = requests.get(api, timeout=10)
        snap = r.json().get("archived_snapshots", {}).get("closest", {})
        if not snap.get("available"):
            return ""
        time.sleep(0.4)
        return get_title_direct(snap["url"])
    except:
        return ""

BAD_TITLES = {
    "barrons.com", "marketwatch.com", "reuters.com",
    "access to this page has been denied",
    "404 - file or directory not found.",
    "just a moment", "403 forbidden"
}

def get_title_smart(url: str, year: int, date_str: str) -> str:
    # Zawsze próbuj bezpośrednio
    title = get_title_direct(url)
    if len(title) > 15 and title.lower() not in BAD_TITLES:
        return title
    # Fallback Wayback Machine tylko dla starszych lat (linki martwe)
    if year <= 2021:
        title = get_title_wayback(url, date_str)
        if len(title) > 15 and title.lower() not in BAD_TITLES:
            return title
    return ""  # zostanie zastąpiony slugiem

def clean_title(title: str) -> str:
    if not title:
        return ""
    title = re.sub(r"^\d{6,7}\s+", "", title)            # ID SeekingAlpha
    title = re.sub(r"\s+\d{4}\s+\d{2}\s+\d{2}$", "", title)  # data MarketWatch
    title = re.sub(r"\s+\d{6,7}$", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    if any(b in title.lower() for b in BAD_TITLES):
        return ""
    return title


def run_title_enrichment():
    """
    Pobiera prawdziwe tytuły i zapisuje do OUTPUT_ENRICHED.
    Plik surowy OUTPUT_RAW pozostaje nienaruszony — backup.
    Checkpoint: pomija URL-e już obecne w OUTPUT_ENRICHED.
    """
    if not os.path.exists(OUTPUT_RAW):
        print("Brak pliku surowego — uruchom najpierw run_full_download()")
        return

    df_raw = pd.read_csv(OUTPUT_RAW)
    df_raw["date"] = pd.to_datetime(df_raw["date"])

    # Wczytaj już wzbogacone — checkpoint tytułów
    if os.path.exists(OUTPUT_ENRICHED):
        df_done   = pd.read_csv(OUTPUT_ENRICHED)
        df_done["date"] = pd.to_datetime(df_done["date"])
        done_urls = set(df_done["url"].tolist())
        print(f"Już wzbogaconych: {len(df_done)} artykułów (wczytano z {OUTPUT_ENRICHED})")
    else:
        df_done   = pd.DataFrame()
        done_urls = set()

    to_fetch = df_raw[~df_raw["url"].isin(done_urls)].copy()
    print(f"Do wzbogacenia:   {len(to_fetch)} / {len(df_raw)}")
    print(f"Backup raw:       {OUTPUT_RAW} (nienaruszony)")
    print(f"Wynik:            {OUTPUT_ENRICHED}")
    print("-" * 50)

    batch = []
    for i, (_, row) in enumerate(to_fetch.iterrows()):
        date_str    = row["date"].strftime("%Y%m%d")
        title       = get_title_smart(row["url"], int(row["year"]), date_str)
        title_clean = clean_title(title)
        # Fallback na slug jeśli nic nie dostaliśmy
        row = row.to_dict()
        row["title"] = title_clean if len(title_clean) > 15 else clean_title(row["title_slug"])
        batch.append(row)

        # Zapisuj co 50 artykułów — checkpoint tytułów
        if len(batch) >= 50:
            df_batch = pd.DataFrame(batch)
            if not df_done.empty:
                df_all = pd.concat([df_done, df_batch], ignore_index=True)
            else:
                df_all = df_batch
            df_all = df_all.drop_duplicates(subset=["url"])
            df_all.to_csv(OUTPUT_ENRICHED, index=False)
            df_done   = df_all
            done_urls = set(df_done["url"].tolist())
            batch     = []
            print(f"  Checkpoint: {i+1}/{len(to_fetch)} | enriched: {len(df_done)}")

        time.sleep(0.6)

    # Zapisz pozostałe
    if batch:
        df_batch = pd.DataFrame(batch)
        if not df_done.empty:
            df_all = pd.concat([df_done, df_batch], ignore_index=True)
        else:
            df_all = df_batch
        df_all = df_all.drop_duplicates(subset=["url"])
        df_all.to_csv(OUTPUT_ENRICHED, index=False)
        df_done = df_all

    # Finalne czyszczenie — usuń rekordy bez tytułu
    df_final = pd.read_csv(OUTPUT_ENRICHED)
    df_final = df_final[df_final["title"].fillna("").str.len() > 15].reset_index(drop=True)
    df_final.to_csv(OUTPUT_ENRICHED, index=False)

    print(f"\n{'='*50}")
    print(f"Gotowe!")
    print(f"  Backup (surowy):    {OUTPUT_RAW}      — {len(df_raw)} rekordów, nienaruszony")
    print(f"  Wzbogacony:         {OUTPUT_ENRICHED} — {len(df_final)} rekordów z tytułami")


# Uruchom po zakończeniu run_full_download()
run_title_enrichment()


Artykułów do wzbogacenia: 8089 / 8089
  Checkpoint: 0/8089


## 4. Klasyfikacja sentymentu — finbert-tone

**Model:** `yiyanghkust/finbert-tone` — wybrany po porównaniu z distilroberta i ProsusAI/finbert.  
Szczegółowy test diagnostyczny w notatniku `NVDA_Sentiment_Pipeline.ipynb`.

**Znane ograniczenia (~15–25% błędów):**
- Idiomy tech: "doubles down" → błędnie Negative
- Slang: "slays keynote" → poprawnie Positive, ale z błędnego powodu
- Dokładność na nagłówkach finansowych: ~75–85% (Malo et al. 2014)


In [4]:
from transformers import pipeline as hf_pipeline

print("Ładowanie finbert-tone...")
clf = hf_pipeline(
    "text-classification",
    model="yiyanghkust/finbert-tone",
    truncation=True,
    max_length=512
)
print("Gotowy!")

def run_finbert(df_news: pd.DataFrame, batch_size: int = 64) -> pd.DataFrame:
    """Klasyfikuje sentyment dla wszystkich tytułów."""
    titles = df_news["title"].tolist()
    all_preds = []

    for i in range(0, len(titles), batch_size):
        batch = titles[i : i + batch_size]
        preds = clf(batch)
        all_preds.extend(preds)
        if i % 500 == 0:
            print(f"  {i}/{len(titles)} artykułów...", flush=True)

    # finbert-tone zwraca Positive/Negative/Neutral (duża litera)
    df_news["sentiment_label"] = [p["label"] for p in all_preds]
    df_news["sentiment_score"] = [
         p["score"] if p["label"] == "Positive" else
        -p["score"] if p["label"] == "Negative" else
         0.0
        for p in all_preds
    ]
    return df_news

OUTPUT_ENRICHED = "nvda_news_enriched.csv"  # backup raw, enriched osobno

# Czytamy z wzbogaconego pliku z prawdziwymi tytułami
df_news = pd.read_csv(OUTPUT_ENRICHED)
df_news["date"] = pd.to_datetime(df_news["date"])
df_news = df_news[df_news["title"].fillna("").str.len() > 15].reset_index(drop=True)

print(f"Artykułów do klasyfikacji: {len(df_news)}")
print(f"Zakres dat: {df_news['date'].min().date()} → {df_news['date'].max().date()}")
print(f"Per rok:")
print(df_news.groupby("year")["title"].count().to_string())

df_news = run_finbert(df_news)

# Zapisz wyniki FinBERT — żeby nie klasyfikować ponownie po restarcie
df_news.to_csv("nvda_news_with_sentiment.csv", index=False)

print("\nRozkład klas:")
print(df_news["sentiment_label"].value_counts())
print(f"\nPrzykłady:")
print(df_news[["date", "title", "sentiment_label", "sentiment_score"]]
      .sample(10).to_string())


Ładowanie finbert-tone...
Gotowy!
Artykułów do klasyfikacji: 8069
Zakres dat: 2017-01-02 → 2025-12-31
Per rok:
year
2017     435
2018     540
2019     517
2020     413
2021     229
2022     345
2023    1256
2024    2058
2025    2276
  0/8069 artykułów...
  8000/8069 artykułów...

Rozkład klas:
sentiment_label
Neutral     5222
Positive    1793
Negative    1054
Name: count, dtype: int64

Przykłady:
           date                                                                                                                                                                   title sentiment_label  sentiment_score
3219 2023-09-21                                                                                                                                            Bloomberg - Are you a robot?         Neutral         0.000000
3582 2023-11-28                                                     Bitdeer Joins Forces With Nvidia In Cloud Service Launch - Bitdeer Technologies (NASDAQ:BTDR), NVI

## 5. Agregacja dzienna i konstrukcja 5 cech do modelu

| Cecha | Opis | Uzasadnienie |
|---|---|---|
| `sent_day` | Średni score dnia (bez neutralnych) | Bieżący sygnał |
| `sent_5d` | Średnia krocząca 5 dni | Trend tygodniowy |
| `sent_10d` | Średnia krocząca 10 dni | Trend 2-tygodniowy |
| `sent_ewm` | Wykładniczy decay span=5 | Pamięć rynku — starsze newsy ważą mniej |
| `sent_intensity_ewm` | Echo mocnych newsów (\|score\|>0.5) span=10 | Długotrwały efekt przełomowych ogłoszeń |

**Brak lagu:** model działa w trybie end-of-day — na koniec dnia `n` zna wszystkie newsy z dnia `n`
i przewiduje ruch w dniu `n+1` (Bollen et al. 2011, Zhang et al. 2018).


In [5]:
def build_daily_features(df_news: pd.DataFrame) -> pd.DataFrame:
    df_news["date"] = pd.to_datetime(df_news["date"]).dt.normalize()

    # Tylko niezerowe — neutralne nie niosą sygnału inwestycyjnego
    nonzero = df_news[df_news["sentiment_score"] != 0]
    daily_nonzero = (nonzero.groupby("date")["sentiment_score"]
                     .mean().rename("sentiment_nonzero").reset_index())

    # Mocne sygnały (|score| > 0.5)
    strong = df_news[df_news["sentiment_score"].abs() > 0.5]
    daily_strong = (strong.groupby("date")["sentiment_score"]
                    .mean().rename("sentiment_intensity").reset_index())

    daily_count = (df_news.groupby("date")["sentiment_score"]
                   .count().rename("news_count").reset_index())

    daily = (daily_count
             .merge(daily_nonzero, on="date", how="left")
             .merge(daily_strong,  on="date", how="left")
             .sort_values("date").reset_index(drop=True))

    daily["sentiment_nonzero"]   = daily["sentiment_nonzero"].fillna(0)
    daily["sentiment_intensity"] = daily["sentiment_intensity"].fillna(0)

    # 5 cech
    daily["sent_day"]  = daily["sentiment_nonzero"]
    daily["sent_5d"]   = daily["sentiment_nonzero"].rolling(5,  min_periods=1).mean()
    daily["sent_10d"]  = daily["sentiment_nonzero"].rolling(10, min_periods=1).mean()
    daily["sent_ewm"]  = daily["sentiment_nonzero"].ewm(span=5,  adjust=False).mean()
    daily["sent_intensity_ewm"] = (daily["sentiment_intensity"]
                                   .ewm(span=10, adjust=False).mean())
    daily["has_news"]  = (daily["news_count"] > 0).astype(int)
    daily["year"]      = daily["date"].dt.year
    daily["month"]     = daily["date"].dt.month

    return daily


MODEL_COLS = ["sent_day", "sent_5d", "sent_10d", "sent_ewm", "sent_intensity_ewm"]

daily_sentiment = build_daily_features(df_news)

# Zapisz
daily_sentiment[["date"] + MODEL_COLS + ["has_news", "news_count"]].to_csv(
    OUTPUT_SENT, index=False
)
print(f"Zapisano: {OUTPUT_SENT}")
print(f"Dni z newsami: {daily_sentiment['has_news'].sum()} / {len(daily_sentiment)}")
print(daily_sentiment[["date"] + MODEL_COLS].tail(10).to_string())


Zapisano: nvda_sentiment_features.csv
Dni z newsami: 1891 / 1891
           date  sent_day   sent_5d  sent_10d  sent_ewm  sent_intensity_ewm
1881 2025-12-19  0.861662  0.131028  0.304817  0.315323            0.257670
1882 2025-12-21  0.999996  0.528365  0.308017  0.543547            0.392639
1883 2025-12-22 -0.979294  0.167731  0.130329  0.035934            0.143197
1884 2025-12-23  0.000000  0.167731  0.068569  0.023956            0.117161
1885 2025-12-24  0.978704  0.372214  0.066440  0.342205            0.273805
1886 2025-12-25 -0.999986 -0.000116  0.065456 -0.105192            0.042207
1887 2025-12-26  0.000000 -0.200115  0.164125 -0.070128            0.034533
1888 2025-12-29  0.000000 -0.004256  0.081737 -0.046752            0.028254
1889 2025-12-30  0.786327  0.153009  0.160370  0.230941            0.166085
1890 2025-12-31  0.998680  0.157004  0.264609  0.486854            0.317466


## 6. Statystyki i analiza sentymentu 2017–2026

Analiza pokrycia, rozkładu sentymentu i kluczowych momentów rynkowych.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_news["date"] = pd.to_datetime(df_news["date"])
df_news["year"]  = df_news["date"].dt.year
df_news["month"] = df_news["date"].dt.to_period("M")

# ── Statystyki per rok ────────────────────────────────────────────────────────
yearly = df_news.groupby("year").agg(
    artykuly        = ("title", "count"),
    pozytywne_pct   = ("sentiment_label", lambda x: (x == "Positive").mean() * 100),
    negatywne_pct   = ("sentiment_label", lambda x: (x == "Negative").mean() * 100),
    neutralne_pct   = ("sentiment_label", lambda x: (x == "Neutral").mean()  * 100),
    avg_score       = ("sentiment_score", "mean"),
    top_source      = ("source", lambda x: x.value_counts().index[0] if len(x) > 0 else ""),
).round(1)

print("="*70)
print("STATYSTYKI PER ROK")
print("="*70)
print(yearly.to_string())

# ── Statystyki per miesiąc (top 5 najbardziej pozytywnych i negatywnych) ──────
monthly = df_news.groupby("month").agg(
    artykuly  = ("title", "count"),
    avg_score = ("sentiment_score", "mean"),
).reset_index()
monthly["month_dt"] = monthly["month"].dt.to_timestamp()

print("\n" + "="*70)
print("TOP 5 NAJBARDZIEJ POZYTYWNYCH MIESIĘCY")
print("="*70)
print(monthly.nlargest(5, "avg_score")[["month", "artykuly", "avg_score"]].to_string())

print("\n" + "="*70)
print("TOP 5 NAJBARDZIEJ NEGATYWNYCH MIESIĘCY")
print("="*70)
print(monthly.nsmallest(5, "avg_score")[["month", "artykuly", "avg_score"]].to_string())

# ── Rozkład źródeł per era ────────────────────────────────────────────────────
print("\n" + "="*70)
print("TOP ŹRÓDŁA 2017–2021 (historyczne)")
print("="*70)
hist = df_news[df_news["year"] <= 2021]
print(hist["source"].value_counts().head(8).to_string())

print("\n" + "="*70)
print("TOP ŹRÓDŁA 2022–2026 (najnowsze, węższe filtrowanie)")
print("="*70)
rec = df_news[df_news["year"] >= 2022]
print(rec["source"].value_counts().head(8).to_string())


## 7. Wizualizacje

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(4, 1, figsize=(16, 18))

# ── Wykres 1: Sentyment miesięczny 2017–2026 ─────────────────────────────────
colors_m = ["#2ecc71" if x > 0.05 else "#e74c3c" if x < -0.05 else "#95a5a6"
            for x in monthly["avg_score"]]
axes[0].bar(monthly["month_dt"], monthly["avg_score"],
            color=colors_m, alpha=0.8, width=20)
axes[0].axhline(0, color="black", linewidth=0.8, linestyle="--")

# Adnotacje kluczowych wydarzeń
events = {
    "2018-10": ("Krach Q4\n2018", "red"),
    "2020-03": ("COVID\nMarzec 2020", "red"),
    "2021-11": ("ATH\nListopad 2021", "green"),
    "2022-10": ("Dno bessy\n2022", "red"),
    "2023-05": ("Boom AI\nMaj 2023", "green"),
    "2024-06": ("H100/B200\nSplit 10:1", "green"),
}
for date_str, (label, color) in events.items():
    try:
        dt = pd.to_datetime(date_str)
        row = monthly[monthly["month_dt"].dt.to_period("M") ==
                      pd.Period(date_str, "M")]
        if not row.empty:
            y = row["avg_score"].values[0]
            axes[0].annotate(label, xy=(dt, y),
                           xytext=(dt, y + (0.06 if y >= 0 else -0.08)),
                           fontsize=7, ha="center", color=color,
                           arrowprops=dict(arrowstyle="->", color=color, lw=0.8))
    except:
        pass

axes[0].set_title("Miesięczny sentyment newsów NVDA 2017–2026 (finbert-tone)", fontsize=13)
axes[0].set_ylabel("Średni score [-1, +1]")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[0].grid(alpha=0.3)

# ── Wykres 2: Liczba artykułów per miesiąc ───────────────────────────────────
axes[1].bar(monthly["month_dt"], monthly["artykuly"],
            color="steelblue", alpha=0.7, width=20)
axes[1].set_title("Liczba artykułów per miesiąc")
axes[1].set_ylabel("Artykuły")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[1].grid(alpha=0.3)

# ── Wykres 3: Rozkład klas per rok (stacked bar) ─────────────────────────────
yearly_plot = df_news.groupby("year")["sentiment_label"].value_counts(normalize=True).unstack(fill_value=0) * 100
yearly_plot = yearly_plot.reindex(columns=["Positive", "Neutral", "Negative"], fill_value=0)
yearly_plot.plot(kind="bar", stacked=True, ax=axes[2],
                 color=["#2ecc71", "#95a5a6", "#e74c3c"], alpha=0.85)
axes[2].set_title("Rozkład sentymentu per rok (%)")
axes[2].set_ylabel("Procent artykułów")
axes[2].set_xlabel("")
axes[2].legend(loc="upper left")
axes[2].grid(alpha=0.3, axis="y")

# ── Wykres 4: EWM decay — "pamięć rynku" ─────────────────────────────────────
daily_sentiment["date"] = pd.to_datetime(daily_sentiment["date"])
axes[3].fill_between(daily_sentiment["date"], daily_sentiment["sent_ewm"],
                     0, where=daily_sentiment["sent_ewm"] >= 0,
                     color="#2ecc71", alpha=0.4, label="Pozytywny")
axes[3].fill_between(daily_sentiment["date"], daily_sentiment["sent_ewm"],
                     0, where=daily_sentiment["sent_ewm"] < 0,
                     color="#e74c3c", alpha=0.4, label="Negatywny")
axes[3].plot(daily_sentiment["date"], daily_sentiment["sent_ewm"],
             color="navy", linewidth=1, alpha=0.8)
axes[3].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[3].set_title("Sentyment dzienny z wykładniczym decay (span=5) — 'pamięć rynku'")
axes[3].set_ylabel("EWM score")
axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[3].legend()
axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("nvda_sentiment_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Wykres zapisany: nvda_sentiment_analysis.png")


## 8. Merge z głównym notatnikiem (df_daily)

Wklej poniższy fragment do notatnika `Nvidia-Copy3.ipynb` po sekcji feature engineering.


In [ ]:
# ── Wklej do głównego notatnika Nvidia-Copy3.ipynb ───────────────────────────
#
# import pandas as pd
#
# MODEL_COLS = ["sent_day", "sent_5d", "sent_10d", "sent_ewm", "sent_intensity_ewm"]
#
# df_sent = pd.read_csv("nvda_sentiment_features.csv")
# df_sent["date"] = pd.to_datetime(df_sent["date"])
#
# # df_daily ma DatetimeIndex
# df_daily["date"] = pd.to_datetime(df_daily.index).normalize()
#
# df_daily = df_daily.merge(df_sent[["date"] + MODEL_COLS + ["has_news", "news_count"]],
#                           on="date", how="left")
#
# # Dni bez newsów → 0 (brak sygnału = neutralny)
# df_daily[MODEL_COLS + ["has_news", "news_count"]] = (
#     df_daily[MODEL_COLS + ["has_news", "news_count"]].fillna(0)
# )
#
# # Usuń kolumnę pomocniczą
# df_daily = df_daily.drop(columns=["date"])
#
# print(f"Shape po merge: {df_daily.shape}")
# print(f"Nowe cechy sentymentu: {MODEL_COLS + ['has_news']}")
# print(df_daily[MODEL_COLS].describe())

print("Szablon gotowy — skopiuj zakomentowany kod do głównego notatnika.")
print(f"Plik z cechami: {OUTPUT_SENT}")
